In [4]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [5]:
df = pd.read_csv("data/ECG_10s_2500.csv")

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Bidirectional, LSTM, Dense, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score


tf.random.set_seed(42)
np.random.seed(42)

In [7]:
patients = df['patient_id'].unique()

valid_patients = []
for pid in patients:
    labels = df[df['patient_id'] == pid]['label'].unique()
    if len(labels) > 1:
        valid_patients.append(pid)

print("✅ Patients usable for testing:", valid_patients)
print("❌ Patients with only normal signals (kept in training):", set(patients) - set(valid_patients))


✅ Patients usable for testing: [np.int64(1), np.int64(2), np.int64(4), np.int64(7), np.int64(8)]
❌ Patients with only normal signals (kept in training): {np.int64(9), np.int64(3), np.int64(5), np.int64(6)}


In [8]:
results = []

for test_patient in valid_patients:
    print(f"\n=== Testing on Patient {test_patient} ===")

    # Split
    train_df = df[df['patient_id'] != test_patient]
    val_df   = df[df['patient_id'] == test_patient]

    # Drop non-signal columns
    X_train = train_df.drop(['label', 'patient_id', 'day', 'count', 'glucose_values'], axis=1).values
    X_val   = val_df.drop(['label', 'patient_id', 'day', 'count', 'glucose_values'], axis=1).values
    y_train = train_df['label'].values
    y_val   = val_df['label'].values

    # Normalize (fit only on training data)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    # Reshape (samples, timesteps, channels)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_val   = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)

    # Compute class weights
    cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights = dict(enumerate(cw))

    # === Model ===
    model = Sequential([
        Input(shape=(X_train.shape[1], 1)),
        Conv1D(32, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Conv1D(64, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Bidirectional(LSTM(64, return_sequences=False, dropout=0.3)),
        Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')]
    )

    # === Training ===
    es = EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        class_weight=class_weights,
        callbacks=[es],
        verbose=1
    )

    # === Evaluation ===
    y_pred_prob = model.predict(X_val)
    y_pred = (y_pred_prob > 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_prob)
    rec = recall_score(y_val, y_pred)

    print(f"✅ Test Accuracy: {acc:.3f} | AUC: {auc:.3f} | Recall: {rec:.3f}")
    results.append((test_patient, acc, auc, rec))



=== Testing on Patient 1 ===
Epoch 1/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 609s 859ms/step - accuracy: 0.5357 - auc: 0.5600 - loss: 0.6970 - recall: 0.5451 - val_accuracy: 0.1511 - val_auc: 0.6128 - val_loss: 0.8260 - val_recall: 0.9444
Epoch 2/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 620s 880ms/step - accuracy: 0.5215 - auc: 0.5900 - loss: 0.6883 - recall: 0.6190 - val_accuracy: 0.2285 - val_auc: 0.6631 - val_loss: 0.7845 - val_recall: 0.9259
Epoch 3/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 640s 908ms/step - accuracy: 0.5700 - auc: 0.6466 - loss: 0.6659 - recall: 0.6464 - val_accuracy: 0.4060 - val_auc: 0.6702 - val_loss: 0.8820 - val_recall: 0.8333
Epoch 4/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 682s 967ms/step - accuracy: 0.6360 - auc: 0.7186 - loss: 0.6215 - recall: 0.7009 - val_accuracy: 0.8998 - val_auc: 0.6029 - val_loss: 0.4751 - val_recall: 0.0556
Epoch 5/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 669s 948ms/step - accuracy: 0.6404 - auc: 0.7576 - loss: 0.5832 - recall: 0.7823 - val_accuracy: 0.9016 - val_auc: 0.6588 

In [ ]:
results = []

for test_patient in valid_patients:
    print(f"\n=== Testing on Patient {test_patient} ===")

    # Split
    train_df = df[df['patient_id'] != test_patient]
    val_df   = df[df['patient_id'] == test_patient]

    # Drop non-signal columns
    X_train = train_df.drop(['label', 'patient_id', 'day', 'count', 'glucose_values'], axis=1).values
    X_val   = val_df.drop(['label', 'patient_id', 'day', 'count', 'glucose_values'], axis=1).values
    y_train = train_df['label'].values
    y_val   = val_df['label'].values

    # Normalize (fit only on training data)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    # Reshape (samples, timesteps, channels)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_val   = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)

    # Compute class weights
    cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights = dict(enumerate(cw))

    # === Model ===
    model = Sequential([
        Input(shape=(X_train.shape[1], 1)),
        Conv1D(64, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.35),

        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.35),

        Bidirectional(LSTM(128, return_sequences=True, dropout=0.35)),
        Bidirectional(LSTM(64, return_sequences=False, dropout=0.35)),
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        Dropout(0.35),
        Dense(64, activation='relu'),
        Dropout(0.35),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')]
    )

    # === Training ===
    es = EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_data=(X_val, y_val),
        class_weight=class_weights,
        callbacks=[es],
        verbose=1
    )

    # === Evaluation ===
    y_pred_prob = model.predict(X_val)
    y_pred = (y_pred_prob > 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_prob)
    rec = recall_score(y_val, y_pred)

    print(f"✅ Test Accuracy: {acc:.3f} | AUC: {auc:.3f} | Recall: {rec:.3f}")
    results.append((test_patient, acc, auc, rec))



=== Testing on Patient 1 ===
Epoch 1/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 2544s 4s/step - accuracy: 0.4612 - auc: 0.5615 - loss: 0.6999 - recall: 0.6365 - val_accuracy: 0.1494 - val_auc: 0.6041 - val_loss: 0.7479 - val_recall: 0.8889
Epoch 2/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 2440s 3s/step - accuracy: 0.4775 - auc: 0.5944 - loss: 0.6848 - recall: 0.6660 - val_accuracy: 0.4077 - val_auc: 0.5950 - val_loss: 0.7113 - val_recall: 0.7407
Epoch 3/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 2546s 4s/step - accuracy: 0.4882 - auc: 0.6763 - loss: 0.6395 - recall: 0.7985 - val_accuracy: 0.7452 - val_auc: 0.7078 - val_loss: 0.5216 - val_recall: 0.5556
Epoch 4/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 2735s 4s/step - accuracy: 0.5327 - auc: 0.7358 - loss: 0.5852 - recall: 0.8646 - val_accuracy: 0.8401 - val_auc: 0.6360 - val_loss: 0.5847 - val_recall: 0.1852
Epoch 5/30
705/705 ━━━━━━━━━━━━━━━━━━━━ 2612s 4s/step - accuracy: 0.5518 - auc: 0.7547 - loss: 0.5638 - recall: 0.8779 - val_accuracy: 0.9016 - val_auc: 0.6025 - val_loss